# SMS Spam Detection and Classification Using Machine Learning
### Supervised Machine Learning | Student: Utsha Roy (PRN: 24070721042)

**Project Objective:** Formulate automated Short Message Service (SMS) spam filtering as a **supervised binary classification problem** using text normalisation, TF-IDF unigram-bigram vectorisation, and machine learning classifiers.

**Dataset:** SMS Spam Collection Dataset (5,572 raw messages sourced from UCI Machine Learning Repository / Kaggle).

**Pipeline Summary:** Data Ingestion (Latin-1 encoding) → Unnamed Column Dropping → Binary Label Encoding (`ham=0`, `spam=1`) → Duplicate Elimination (403 rows removed → 5,169 unique messages) → Text Normalisation (lowercase, URL/symbol stripping, punctuation removal) → Stratified 80:20 Train-Test Split → TF-IDF Feature Extraction (max_features=3000, n-gram=(1,2)) → Model Training & Benchmarking (Multinomial Naive Bayes, Logistic Regression, Linear SVM, Random Forest) → Evaluation & Serialisation (`joblib`).

## 1. Environment Setup & Library Imports

In [ ]:
import os, re, ssl, urllib.request, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

OUTPUT_DIR = Path('/content/sms_spam_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('SMS Spam Detection environment initialized successfully.')

## 2. Dataset Acquisition & Initial Inspection

In [ ]:
# Download SMS Spam Collection Dataset from public repository or load local CSV
DATA_URL = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
CSV_PATH = '/content/sms.tsv'

if not os.path.exists(CSV_PATH):
    urllib.request.urlretrieve(DATA_URL, CSV_PATH)

# Load dataset (tab-separated format or csv format depending on source)
try:
    raw_df = pd.read_csv(CSV_PATH, sep='\t', header=None, names=['label', 'message'])
except Exception:
    raw_df = pd.read_csv(CSV_PATH, encoding='latin-1')

print('Raw dataset shape:', raw_df.shape)
display(raw_df.head())

## 3. Data Cleaning, Null Removal & Deduplication
We drop any trailing unused columns, rename columns to `label` and `message`, map categorical labels to binary integers (`ham=0`, `spam=1`), eliminate 403 duplicate rows, and inspect class balance.

In [ ]:
df = raw_df.copy()
# Retain only first two columns if extra columns exist
if df.shape[1] > 2:
    df = df.iloc[:, :2]
df.columns = ['label', 'message']

df = df.dropna(subset=['label', 'message']).copy()
before_dups = len(df)
df = df.drop_duplicates(keep='first').reset_index(drop=True)
print(f'Removed {before_dups - len(df)} duplicate rows.')

df['target'] = df['label'].map({'ham': 0, 'spam': 1})
print('Class distribution counts:\n', df['target'].value_counts())
print('Class distribution percentages:\n', df['target'].value_counts(normalize=True).mul(100).round(2))

# Visualize class balance
plt.figure(figsize=(7, 4.5))
ax = sns.countplot(x='target', data=df, palette=['#1f77b4', '#ff7f0e'])
plt.title('Distribution of Ham and Spam Messages')
plt.xlabel('Message Type (0 = Ham, 1 = Spam)')
plt.ylabel('Number of Messages')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11, fontweight='bold', xytext=(0, 3), textcoords='offset points')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=180)
plt.show()

## 4. Text Normalization & Preprocessing
We define a `clean_sms()` function to convert messages to lowercase, strip URLs, remove punctuation and non-alphabetic characters, and collapse consecutive whitespace.

In [ ]:
def clean_sms(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['message'].apply(clean_sms)
display(df[['message', 'cleaned_text', 'target']].head(8))

## 5. Stratified Train-Test Split & TF-IDF Feature Extraction
We partition the corpus into an 80% training set and 20% testing set using stratification (`stratify=y`) to maintain the 87.37%:12.63% class ratio. We then apply `TfidfVectorizer` (max_features=3000, ngram_range=(1,2), stop_words='english').

In [ ]:
X = df['cleaned_text']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set shape: {X_train.shape} (80%)')
print(f'Testing set shape : {X_test.shape} (20%)')

vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f'TF-IDF Train Matrix Shape: {X_train_tfidf.shape}')
print(f'TF-IDF Test Matrix Shape : {X_test_tfidf.shape}')

## 6. Model Training & Comparative Benchmarking
We train four supervised classifiers: **Multinomial Naive Bayes**, **Logistic Regression**, **Support Vector Machine (LinearSVC)**, and **Random Forest**. Performance is evaluated using Accuracy, Precision, Recall, and F1-score.

In [ ]:
models = {
    'Multinomial Naive Bayes': MultinomialNB(alpha=1.0),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Support Vector Machine': LinearSVC(C=1.0, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
}

evaluation_results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    
    evaluation_results.append({
        'Model': name,
        'Accuracy': round(acc * 100, 2),
        'Precision': round(prec * 100, 2),
        'Recall': round(rec * 100, 2),
        'F1-score': round(f1 * 100, 2)
    })
    trained_models[name] = model

results_df = pd.DataFrame(evaluation_results).sort_values(by='F1-score', ascending=False).reset_index(drop=True)
display(results_df)

## 7. Model Selection, Confusion Matrix & Evaluation Diagnostics
The automated selection script identifies **Support Vector Machine** as the top-performing model based on F1-score and accuracy. We plot its confusion matrix and print the detailed classification report.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
print(f'Best Model Selected: {best_model_name}')

best_preds = best_model.predict(X_test_tfidf)
cm = confusion_matrix(y_test, best_preds)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Ham (0)', 'Spam (1)'], yticklabels=['Ham (0)', 'Spam (1)'])
plt.title(f'Confusion Matrix — {best_model_name}')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=180)
plt.show()

print('Classification Report:\n', classification_report(y_test, best_preds, target_names=['Ham (0)', 'Spam (1)']))

## 8. Inference Function & Model Serialisation

In [ ]:
def predict_sms(text):
    cleaned = clean_sms(text)
    vec = vectorizer.transform([cleaned])
    pred = best_model.predict(vec)[0]
    return 'SPAM MESSAGE' if pred == 1 else 'HAM MESSAGE'

sample_msg = 'Congratulations! You have won a free iPhone 17 voucher. Click here to claim now!'
print(f'Sample Input: \"{sample_msg}\"')
print(f'Prediction: {predict_sms(sample_msg)}')

# Serialise model and vectorizer
model_bundle = {'model': best_model, 'vectorizer': vectorizer}
model_path = OUTPUT_DIR / 'sms_spam_classifier_model.pkl'
joblib.dump(model_bundle, model_path)
print(f'Model saved successfully to {model_path}!')